# Week 3 Seminar 

## Questions? 

We are going to discuss noise and combining images in idealised cases

We are going to look at the steps of image processing using numpy to reveal the mathematical operations

We are going to look at image stats and what they measure - is it structure or is it noise? 


In [ ]:
# Import various libraries 

import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import gc                               



# Noise, not noise and standard deviation

### Let's create an 2D list / image of Gaussian random noise and display it 

In [ ]:
randimage=np.random.normal(loc=0.0, scale=100.0, size=(1024,2048))  # Comment 

In [ ]:
### Print and display the random noise 
print(randimage)

### Should be familiar now - what am I doing here?
vmin=np.nanpercentile(randimage, 1)
vmax=np.nanpercentile(randimage, 99)
plt.imshow(randimage, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
plt.title('Random noise image')
plt.xlabel('x')
plt.xlabel('y')
plt.show()


### Question: What is being plotted below?

In [ ]:

plx=np.linspace(0.0, len(randimage[0])-1, num=len(randimage[0])) # What is this? What does linspace do
ply=randimage[10,:]                                              # What does this correspond to?
plt.plot(plx, ply) 
plt.title('Title')
plt.xlabel('x')
plt.xlabel('y')

### Question: Run the stats - how do they compare to expectations?

### <font color='blue'> Answer: </font>


In [ ]:
print('Median :', np.nanmedian(randimage))
print('Standard deviatiom: ', np.std(randimage))

### Question: Could you also estimate the standard deviation from percentiles? If so what circumstances must apply?

### <font color='blue'> Answer: </font>


### Lets create a version of the image with sorted values along the x-axis 


In [ ]:
sortimage=np.sort(randimage)
vmin=np.nanpercentile(sortimage, 1)
vmax=np.nanpercentile(sortimage, 99)

plt.imshow(sortimage, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
plt.title('Sorted data')
plt.xlabel('x')
plt.xlabel('y')
plt.show()


## Question: What is being plotted below? 

### <font color='blue'> Answer: </font>


In [ ]:

plx=np.linspace(0.0, len(sortimage[0])-1, num=len(sortimage[0])) # What is this? What does linspace do
ply=sortimage[10,:]                                              # What does this correspond to?
plt.plot(plx, ply) 
plt.title('Title')
plt.xlabel('x')
plt.xlabel('y')


## Question: Is the sorted image purely random noise anymore? 

### <font color='blue'> Answer: </font>


## Question: Run the stats on the sorted image - how do they compare to what we determined earlier?

### <font color='blue'> Answer: </font>


In [ ]:
print('Median :', np.nanmedian(sortimage))
print('Standard deviatiom: ', np.std(sortimage))

# Noise and combining images 

Lets create a bunch of images just containing Gaussian random noise

Lets then combine them together 


In [ ]:
images = []  # empty list of images

# Create random images and add them to the list until I have 10 images
while len(images)<10:
    randimage=np.random.normal(loc=0.0, scale=100.0, size=(1024,1024))
    images.append(randimage)

vmin=np.nanpercentile(images[0], 1)
vmax=np.nanpercentile(images[0], 99)

for idx, image in enumerate(images):
    plt.rcParams["figure.figsize"] = (2,2)
    plt.imshow(image[0:20,0:20], vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
    title = 'Subset of image ' + str(idx)
    plt.title(title)
    plt.xlabel('x')
    plt.xlabel('y')
    print('Image ' + str(idx))
    print('Median :', np.nanmedian(image))
    print('Standard deviation: ', np.std(image))
    plt.show()


## Let's do combine operations in numpy 

In [ ]:
# Create initial values that are just copies of the first image 
avimage = images[0].copy()
medimage = images[0].copy()

# Step through the pixels (a bit inefficient by OK)
yi=0
while yi<len(medimage):
    xi=0
    while xi<len(medimage[0]):
        t=[]
        for image in images:
            t.append(image[yi, xi])
        avimage[yi,xi]=np.nanmean(t)
        medimage[yi,xi]=np.nanmedian(t)
        xi=xi+1
    yi=yi+1

In [ ]:
## Let's do the stats and display the images 


In [ ]:

# Why scale to the first image in the list?
vmin=np.nanpercentile(images[0], 1)
vmax=np.nanpercentile(images[0], 99)

# Reuse code from earlier to display
image=avimage
plt.rcParams["figure.figsize"] = (2,2)
plt.imshow(image[0:20,0:20], vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
title = 'Subset of average '
plt.title(title)
plt.xlabel('x')
plt.xlabel('y')
print('Average')
print('Median :', np.nanmedian(image))
print('Standard deviation: ', np.std(image))
plt.show()

image=medimage
plt.rcParams["figure.figsize"] = (2,2)
plt.imshow(image[0:20,0:20], vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
title = 'Subset of median '
plt.title(title)
plt.xlabel('x')
plt.xlabel('y')
print('Median')
print('Median :', np.nanmedian(image))
print('Standard deviation: ', np.std(image))
plt.show()


## Question: How do the average and median images compare visually?

### <font color='blue'> Answer: </font>

## Question: How do the stats compare? Is there an expected value?

### <font color='blue'> Answer: </font>


# Basic data processing as mathematical operations 

We can actually do the data processing done by ccdproc as numpy operations

Advantage is we see the operations at work directly

## Question: What could be a disadvantage of using numpy rather than ccdproc?

Here we are using fits.open to open the images 

It assumes multi-extension FITS files (i.e. lists) with our data typically at index zero

fits.open has the advantage of being a core astropy package .

fits.open can open online data and files zipped in different formats (zip, gz, bz2)


In [ ]:

bias = fits.open('bias_median.fits')

dark = fits.open('dark_median.fits')

flat = fits.open('Flat_R_median.fits')

scimage = fits.open('NGC_3766_R_00004556.fits')


In [ ]:
vmin=np.nanpercentile(scimage[0].data, 5)
vmax=np.nanpercentile(scimage[0].data, 95)
plt.rcParams["figure.figsize"] = (6,6)
plt.imshow(scimage[0].data, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
title = 'Scimage '
plt.title(title)
plt.xlabel('x')
plt.xlabel('y')
plt.colorbar(shrink=0.65)
print('Average')
print('Median :', np.nanmedian(image))
print('Standard deviation: ', np.std(image))
plt.show()


## Lets step through the data processing with bias subtraction


In [ ]:
tempimage=scimage[0].copy()  # What am I doing here?

# Bias subtraction - am I doing it right?
tempimage.data = tempimage.data - bias[0].data

vmin=np.nanpercentile(tempimage.data, 5)
vmax=np.nanpercentile(tempimage.data, 95)
plt.rcParams["figure.figsize"] = (6,6)
plt.imshow(tempimage.data, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
title = 'Tempimage '
plt.title(title)
plt.xlabel('x')
plt.xlabel('y')
plt.colorbar(shrink=0.65)
print('Average')
print('Median :', np.nanmedian(image))
print('Standard deviation: ', np.std(image))
plt.show()


### Question: Has the bias subtraction worked? How do the numbers compare with expectation?

### <font color='blue'> Answer: </font>


In [ ]:
## Lets step through the data processing with dark subtraction


In [ ]:
# Dark subtraction - am I doing it right?
tempimage.data = tempimage.data - dark[0].data 

vmin=np.nanpercentile(tempimage.data, 5)
vmax=np.nanpercentile(tempimage.data, 95)
plt.rcParams["figure.figsize"] = (6,6)
plt.imshow(tempimage.data, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
title = 'Tempimage '
plt.title(title)
plt.xlabel('x')
plt.xlabel('y')
plt.colorbar(shrink=0.65)
print('Average')
print('Median :', np.nanmedian(image))
print('Standard deviation: ', np.std(image))
plt.show()


### Question: Has the dark subtraction worked? How do the numbers compare with expectation?

### <font color='blue'> Answer: </font>


### Lets do the flat field correction

### Question: We normalise the flat - why do we do that? 

### <font color='blue'> Answer: </font>


In [ ]:
# Flat - am I doing it right
tempimage.data = tempimage.data / flat[0].data 

vmin=np.nanpercentile(tempimage.data, 5)
vmax=np.nanpercentile(tempimage.data, 95)
plt.rcParams["figure.figsize"] = (6,6)
plt.imshow(tempimage.data, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
title = 'Tempimage '
plt.title(title)
plt.xlabel('x')
plt.xlabel('y')
plt.colorbar(shrink=0.65)
print('Average')
print('Median :', np.nanmedian(image))
print('Standard deviation: ', np.std(image))
plt.show()

# Callback - stats, biases and flats 

## Question: How could we visually look at what are the stats of the bias and flat measuring?


In [ ]:
bias.close()
dark.close()
flat.close()
scimage.close()


